In [1]:
from fetch import get_games
games = get_games("adr_mog")
print(len(games))
print(games[0].keys())

53
dict_keys(['url', 'pgn', 'time_control', 'end_time', 'rated', 'accuracies', 'tcn', 'uuid', 'initial_setup', 'fen', 'time_class', 'rules', 'white', 'black', 'eco'])


In [2]:
from fetch import get_games
from parse import games_table, moves_table

games = get_games("adr_mog")
gdf = games_table(games, "adr_mog")
moves = moves_table(gdf)
gdf.head()

,game_id,end_time,time_class,base,inc,color,rating,opp_rating,opponent,result,opp_result,score,opening,pgn
0,184675215370,2026-10-01 21:30:34,rapid,600,0,black,965,769,stepanorsah,win,checkmated,1.0,French Defense Winawer Fingerslip Variation,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
1,184675909066,2026-10-01 21:54:45,rapid,600,0,white,1068,811,ayodejiii,win,checkmated,1.0,English Opening Reversed Sicilian Variation,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
2,184679288762,2026-10-01 23:48:30,rapid,600,0,white,961,1060,sergioayala111,resigned,win,0.0,Vienna Game Max Lange Defense 3.Bc4 Bc5 4.d3,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
3,184679544550,2026-10-01 23:55:14,rapid,600,0,black,1041,955,el-tigro56,win,abandoned,1.0,Dutch Defense 2.Nf3,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
4,184679761530,2026-10-02 00:12:07,rapid,600,0,black,975,1041,max1milliano,resigned,win,0.0,Dutch Defense 2.Nf3 Nf6,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."


In [3]:

moves[["ply", "san", "clock", "time_spent"]].head(10)



,ply,san,clock,time_spent
0,1,e4,597.4,2.6
1,2,e6,597.5,2.5
2,3,d4,590.3,7.1
3,4,d5,589.1,8.4
4,5,Nc3,584.3,6.0
5,6,Bb4,567.7,21.4
6,7,Bd2,580.9,3.4
7,8,Nc6,559.0,8.7
8,9,a3,579.3,1.6
9,10,Bxc3,540.3,18.7


In [4]:
moves["clock_min"] = moves["clock"] / 60
moves[["ply", "san", "clock_min", "time_spent"]].head(10)

,ply,san,clock_min,time_spent
0,1,e4,9.956667,2.6
1,2,e6,9.958333,2.5
2,3,d4,9.838333,7.1
3,4,d5,9.818333,8.4
4,5,Nc3,9.738333,6.0
5,6,Bb4,9.461667,21.4
6,7,Bd2,9.681667,3.4
7,8,Nc6,9.316667,8.7
8,9,a3,9.655000,1.6
9,10,Bxc3,9.005000,18.7


In [5]:
from openings import opening_stats, opening_tree, reply_distribution
opening_stats(gdf).head(10)
opening_tree(gdf, moves, "black", depth=4).head(10)
reply_distribution(moves, gdf, ["e4"]) # they're Black, you play 1.e4
reply_distribution(moves, gdf, ["d4", "Nf6"]) # they're White and answered 1.d4 Nf6


Bf4    1.0
Name: proportion, dtype: float64

In [6]:
from openings import opening_stats, opening_tree, reply_distribution
display(opening_stats(gdf).head(10))
display(opening_tree(gdf, moves, "black", depth=4).head(10))
display(reply_distribution(moves, gdf, ["e4"]))
display(reply_distribution(moves, gdf, ["d4", "Nf6"]))

,color,opening,count,sum,raw_rate,smoothed,ci_low,ci_high
0,black,Kings Pawn Opening 1...e5,3,1.0,0.333333,0.490566,0.315602,0.666222


,line,count,sum,smoothed,share
10,e4 e5 Nf3 Nc6,10,8.0,0.668868,0.370370
0,Nf3 f5 e4 e6,1,0.0,0.488851,0.037037
1,d4 d5 Bf4 Nc6,1,0.0,0.488851,0.037037
16,e4 e6 d4 d5,1,1.0,0.579760,0.037037
15,e4 e5 g3 Bc5,1,0.0,0.488851,0.037037
14,e4 e5 d4 d6,1,0.0,0.488851,0.037037
13,e4 e5 Qh5 Nc6,1,0.0,0.488851,0.037037
12,e4 e5 Qg4 Nc6,1,0.0,0.488851,0.037037
11,e4 e5 Qf3 Nf6,1,1.0,0.579760,0.037037
9,e4 e5 Bc4 f5,1,1.0,0.579760,0.037037


e5    0.842105
e6    0.052632
f5    0.052632
c5    0.052632
Name: proportion, dtype: float64

Bf4    1.0
Name: proportion, dtype: float64

In [7]:
from engine_eval import evaluate_games
evals = evaluate_games(moves, "adr_mog", depth=10, max_games=20)
evals[["ply", "san", "cp_before", "cp_after", "wp_loss", "quality"]].head(20)
evals[evals["is_player"]]["quality"].value_counts(normalize=True)


quality
ok            0.864646
inaccuracy    0.066667
blunder       0.039731
mistake       0.028956
Name: proportion, dtype: float64

In [8]:
from features import player_features
player_features(gdf, evals)

{'avg_wp_loss': np.float64(5.189552111975181),
 'blunder_rate_opening': np.float64(0.022988505747126436),
 'blunder_rate_middlegame': np.float64(0.0512396694214876),
 'blunder_rate_endgame': np.float64(0.0446927374301676),
 'blunder_rate_low_clock': np.float64(0.04040404040404041),
 'avg_time_per_move': np.float64(0.019259820426487093),
 'time_trouble_rate': np.float64(0.16981132075471697),
 'timeout_loss_rate': np.float64(0.07547169811320754),
 'capture_rate': np.float64(0.25925925925925924),
 'check_rate': np.float64(0.08754208754208755),
 'avg_game_length': np.float64(55.9622641509434),
 'draw_rate': np.float64(0.018867924528301886),
 'resign_share_of_losses': np.float64(0.5416666666666666),
 'opening_entropy': 1.1047909373235627,
 'score_overall': np.float64(0.5377358490566038),
 'score_after_loss': np.float64(0.5)}

In [9]:
from baseline import pick_baseline_players, build_baseline

print(gdf["time_class"].value_counts())   # which time controls you play

rating = int(gdf["rating"].iloc[-1])
peers = pick_baseline_players(gdf, rating, n=5)
print(peers)
baseline = build_baseline(peers, time_class="rapid")
baseline

time_class
rapid    47
blitz     6
Name: count, dtype: int64
['stepanorsah', 'ayodejiii', 'loadstuff', 'dylanraadestroyer', 'dein02']


,avg_wp_loss,blunder_rate_opening,blunder_rate_middlegame,blunder_rate_endgame,blunder_rate_low_clock,avg_time_per_move,time_trouble_rate,timeout_loss_rate,capture_rate,check_rate,avg_game_length,draw_rate,resign_share_of_losses,opening_entropy,score_overall,score_after_loss
username,,,,,,,,,,,,,,,,
stepanorsah,6.478559,0.026667,0.124294,0.019417,0.000000,0.016990,0.066667,0.000,0.255814,0.148837,57.533333,0.066667,0.000000,-0.000000,0.233333,0.200000
loadstuff,5.157643,0.021563,0.053667,0.024911,NaN,0.007690,0.000000,0.000,0.232865,0.136251,61.974359,0.050000,0.076923,-0.000000,0.650000,0.807692
dylanraadestroyer,5.171307,0.028278,0.070175,0.018727,0.000000,0.012998,0.025000,0.000,0.272986,0.112796,52.750000,0.025000,0.100000,0.286397,0.487500,0.447368
dein02,6.088054,0.082902,0.057866,0.024590,0.009615,0.018379,0.200000,0.125,0.228233,0.092139,59.350000,0.000000,0.307692,0.986427,0.350000,0.400000


In [10]:
from style import zscores
from features import player_features

z = zscores(baseline, player_features(gdf, evals))
z.sort_values()

check_rate                -1.389786
avg_wp_loss               -0.803078
blunder_rate_middlegame   -0.774329
blunder_rate_opening      -0.584699
draw_rate                 -0.567385
avg_game_length           -0.498827
score_after_loss           0.143159
capture_rate               0.565243
score_overall              0.598658
timeout_loss_rate          0.707547
time_trouble_rate          1.087810
avg_time_per_move          1.094248
opening_entropy            1.689803
resign_share_of_losses     3.197756
blunder_rate_low_clock     6.700762
blunder_rate_endgame       6.917638
dtype: float64

In [11]:
import pandas as pd
mine = player_features(gdf, evals)
pd.DataFrame({"you": pd.Series(mine), "baseline_avg": baseline.mean()})

,you,baseline_avg
avg_wp_loss,5.189552,5.723891
blunder_rate_opening,0.022989,0.039852
blunder_rate_middlegame,0.051240,0.076501
blunder_rate_endgame,0.044693,0.021911
blunder_rate_low_clock,0.040404,0.003205
avg_time_per_move,0.019260,0.014015
time_trouble_rate,0.169811,0.072917
timeout_loss_rate,0.075472,0.031250
capture_rate,0.259259,0.247475
check_rate,0.087542,0.122506
